In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import warnings
warnings.filterwarnings('ignore')

# Task 1: Write your code here:
delivery_path = os.path.join(path, 'Q1_data.csv')
df_delivery = pd.read_csv(delivery_path)

In [ ]:
# Task 2: Write your code here:
df_delivery.head()

In [ ]:
# Task 3: Write your code here:
df_delivery.info()

In [ ]:
# Task 4: Write your code here:
df_delivery.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df_delivery['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Target Distribution')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df_delivery.drop(columns=['Order_ID'], inplace = True)

In [ ]:
# Task 2: Write your code here:
#We use this code to figure out the count of missing values int eh various features and plot their graphs to check their distributions
print("Missing values:")
print(df_delivery.isnull().sum())

#Weather Distribution
weather_counts = df_delivery['Weather'].value_counts()
plt.figure(figsize=(10, 5))
plt.bar(weather_counts.index, weather_counts.values, color='teal')
plt.title('Weather Distribution')
plt.xlabel('Weather Type')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show()

#Traffic Level Distribution
traffic_counts = df_delivery['Traffic_Level'].value_counts()
plt.figure(figsize=(10, 5))
plt.bar(traffic_counts.index, traffic_counts.values, color='teal')
plt.title('Traffic Distribution')
plt.xlabel('Traffic Level')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show()

#Time of the Day Distribution
time_counts = df_delivery['Time_of_Day'].value_counts()
plt.figure(figsize=(10, 5))
plt.bar(time_counts.index, time_counts.values, color='teal')
plt.title('Time of Day Distribution')
plt.xlabel('Time of Day')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show()

#Courier experience hours distribution
plt.figure(figsize=(10, 5))
plt.hist(df_delivery['Courier_Experience_yrs'].dropna(), bins=10, edgecolor='black', color='green')
plt.title('Courier_Experience_yrs Distribution')
plt.xlabel('Years')
plt.ylabel('Frequency')
plt.show()

#Delivery Time distribution
plt.figure(figsize=(10, 5))
plt.hist(df_delivery['Delivery_Time'].dropna(), bins=10, edgecolor='black', color='green')
plt.title('Delivery_Time Distribution')
plt.xlabel('Time')
plt.ylabel('Frequency')
plt.show()

# from the graphs we can take the following steps to remove the missing values
df_delivery['Weather'].fillna(df_delivery['Weather'].mode()[0], inplace=True) #fill missing weathers with mode
df_delivery['Traffic_Level'] = df_delivery['Traffic_Level'].fillna('unknown')#fill missing with unknown
df_delivery['Time_of_Day'] = df_delivery['Time_of_Day'].fillna('unknown') #fill missing times with unknown
df_delivery['Courier_Experience_yrs'].fillna(df_delivery['Courier_Experience_yrs'].mean(), inplace=True) #fill with mean
df_delivery.dropna(subset=['Delivery_Time'], inplace=True) #drop rows with missing target




In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_delivery)
#Dropped all dupplicate rows

In [ ]:
# Task 4: Write your code here:
le = LabelEncoder()
df_delivery['Weather'] = le.fit_transform(df_delivery['Weather'])
df_delivery['Traffic_Level'] = le.fit_transform(df_delivery['Traffic_Level'])
df_delivery['Time_of_Day'] = le.fit_transform(df_delivery['Time_of_Day'])
df_delivery['Vehicle_Type'] = le.fit_transform(df_delivery['Vehicle_Type'])

df_delivery.head(10)

In [ ]:
# Task 5: Write your code here:
X = df_delivery.drop("Delivery_Time", axis=1).astype(float)
y = df_delivery['Delivery_Time']


#scaling using standard scalar

features = df_delivery.columns.drop("Delivery_Time")

scaler = StandardScaler()
df_delivery[features] = scaler.fit_transform(df_delivery[features])
df_delivery.head()

In [ ]:
# Task 6: Write your code here:
# target Distribution
plt.figure(figsize=(10, 5))
plt.hist(df_delivery['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Target Distribution')
plt.xlabel('Time')
plt.ylabel('Frequency')
plt.show()

#Delivery time is almost balanced with a few outliers

In [ ]:
# Task 1: Write your code here:
X = df_delivery.drop("Delivery_Time", axis=1).astype(float)
y = df_delivery['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
print("Model trained!")

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)

print(f"MAE:  ${mae:,.2f}")


In [ ]:
# Task 1: Write your code here:
feature_cols = ['Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day',
                'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs']
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(pd.DataFrame(y_pred), bins=50, edgecolor='black')
plt.title('Predicted Delivery Time')
plt.xlabel('Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task Bonus: Write your code here:
%pip install kagglehub catboost lightgbm tqdm -q
from catboost import CatBoostRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score
# Storage for results
all_results = {}

models = {
  "Random Forest Regressor": RandomForestRegressor(n_estimators=200),
  "CatBoost": CatBoostRegressor(verbose=0)
}

for name in models:
  all_results[name] = {'mse': [], 'rmse': [], 'r2': []}
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{5}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    mse = sklearn_mse(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    # Store results
    all_results[model_name]["mse"].append(mse)
    all_results[model_name]["rmse"].append(rmse)
    all_results[model_name]["r2"].append(r2)
    #all_results[model_name]["mae"].append(mae)

for model_name in all_results:
  print(f"\n{model_name}:")
  print(f"  MSE:  {np.mean(all_results[model_name]['mse']):.4f}")
  print(f"  RMSE: {np.mean(all_results[model_name]['rmse']):.4f}")
  print(f"  R2:    {np.mean(all_results[model_name]['r2']):.4f}")
  #print(f"  MAE:    {np.mean(all_results[model_name]['mae']):.4f}")
